# Particle Transformer vs. QUIVER on JetClass (thesis Chapter 6)

This notebook re-runs the jet-tagging models of Chapter 6 on a small demo set and rebuilds the AUC comparison of the
thesis results table: the **Particle Transformer (ParT)** baseline and **QUIVER**, ParT with one extra input token per
particle built from the jet's quantum Fisher information matrix (QFIM), appended to the particle tokens ("PConcat").

| | |
|---|---|
| tasks | Top (t &rarr; bqq&prime;) vs. QCD and H &rarr; cc&#772; vs. QCD (JetClass, 10 leading particles per jet) |
| particle features | **Kin**: 3 (log p<sub>T</sub>/p<sub>T,jet</sub>, &Delta;&eta;, &Delta;&phi;); **Full**: 17 (kinematics, charge, particle ID, impact parameters) |
| training budget *N* | 100K, 500K, 5M jets per epoch (60 epochs, validation 20 % of *N*) |
| models | ParT (2.14 M parameters), QUIVER (2.29 M) |
| checkpoints | 24 = 2 tasks &times; 2 feature sets &times; 3 budgets &times; 2 models; the first of the five thesis seeds (rep 1) each, at its best-validation epoch (`net_best_epoch_state.pt`, the checkpoint the thesis test scores were computed from) |

**Demo set.** 10,000 signal + 10,000 QCD test jets per task (the first 10,000 jets of one signal and one QCD test file),
so the AUCs below scatter by about &plusmn;0.001&ndash;0.002 (bootstrap &sigma; is printed) around the values on the full
3.9 M-jet test set. The table compares them with the thesis AUC of the same checkpoint on the full test set and with the
thesis mean over five seeds. Background rejection at 50 % signal efficiency is not shown: with 10,000 QCD jets it would
rest on only about 10&ndash;200 surviving background jets.

**Requirements.** Python &ge; 3.9 with `torch` (&ge; 2.0), `numpy` and `scikit-learn`; no GPU and no weaver installation
needed. The model code is in `thesis_models/part.py`, the inputs in `data/jetclass_*_demo.npz`, the weights in
`weights/part/`. Inference of all 24 models takes about 6-10 minutes on a 4-core CPU. `tools/export_part.py` documents how the
inputs and weights were produced from the thesis runs.

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import roc_auc_score

from thesis_models import part

HERE = Path.cwd()                      # the notebook folder
DATA = HERE / 'data'
WEIGHTS = HERE / 'weights' / 'part'
DEVICE = 'cpu'                         # 'cuda' works as well
TASKS = {'TTbar': 'Top vs QCD', 'HToCC': 'H->cc vs QCD'}
FILES = {'TTbar': 'jetclass_ttbar_demo.npz', 'HToCC': 'jetclass_htocc_demo.npz'}
BUDGETS, FEATURES, MODELS = ('100K', '500K', '5M'), ('Kin', 'Full'), ('ParT', 'QUIVER')

print(f'torch {torch.__version__}, {torch.get_num_threads()} CPU threads')

## Demo inputs

Each `.npz` file holds, for 20,000 jets (signal first, then QCD), the arrays weaver feeds the networks, i.e. after the
preprocessing of the thesis data configs (`config_*` / `config_*_yaml` inside the file):

* the 10 leading particles of each jet; shorter jets are padded by repeating their particles (`pad_mode: wrap`), and
  `pf_mask` (N, 1, 10) is 1 for real particles and 0 for padding;
* `pf_features_kin` (N, 3, 10): log(p<sub>T</sub>/p<sub>T,jet</sub>) standardized as (x + 4.7)&middot;0.7, &Delta;&eta;, &Delta;&phi; to the jet axis;
* `pf_features_full` (N, 17, 10): log p<sub>T</sub> (x &minus; 1.7)&middot;0.7, log E (x &minus; 2.0)&middot;0.7,
  log p<sub>T</sub>/p<sub>T,jet</sub> and log E/E<sub>jet</sub> (x + 4.7)&middot;0.7, &Delta;R (x &minus; 0.2)&middot;4,
  charge, 5 particle-ID flags, tanh d<sub>0</sub>, &sigma;(d<sub>0</sub>) and tanh d<sub>z</sub>, &sigma;(d<sub>z</sub>)
  (errors clipped to [0, 1]), &Delta;&eta;, &Delta;&phi;; standardized values are clipped to [&minus;5, 5];
* `pf_vectors` (N, 4, 10): (p<sub>x</sub>, p<sub>y</sub>, p<sub>z</sub>, E) in GeV, from which ParT computes its pairwise
  interaction features (ln k<sub>T</sub>, ln z, ln &Delta;, ln m&sup2;); `pf_points` (&Delta;&eta;, &Delta;&phi;) is passed but unused;
* `qfim` (N, 90, 10), QUIVER only: the branches `qf0` &hellip; `qf89` unchanged. QUIVER token *p* is
  (qf<sub>0</sub>[p], &hellip;, qf<sub>89</sub>[p]). Read row-major, the 900 numbers of a jet are its 30 &times; 30 QFIM,
  F[r, s] = qf<sub>c</sub>[p] with 30r + s = 10c + p, so `qfim[n].reshape(30, 30)` is the (symmetric) matrix of jet *n*.
  The stored values are the Fubini&ndash;Study metric g (F<sub>Q</sub> = 4g); the model sees them as stored.

`label` is 1 for signal and 0 for QCD; `jet_pt`, `jet_sdmass`, ... and `source_file` / `source_entry` are for reference.
The QCD jets of the two files are the same 10,000 JetClass jets, but the Top and H &rarr; cc&#772; samples carry separately
produced QFIM branches, so their `qfim` arrays differ.

In [ ]:
demo = {}
for task, fname in FILES.items():
    with np.load(DATA / fname) as z:
        demo[task] = {k: z[k] for k in z.files}
    d = demo[task]
    print(f"{TASKS[task]}: {len(d['label'])} jets ({int(d['label'].sum())} signal), from "
          + ', '.join(f'{Path(f).name}[0:{(d["source_file"] == i).sum()}]' for i, f in enumerate(d['source_files'])))
    for k in ('pf_points', 'pf_features_kin', 'pf_features_full', 'pf_vectors', 'pf_mask', 'qfim'):
        print(f'    {k:17s} {str(d[k].shape):16s} {d[k].dtype}')
    F = d['qfim'].reshape(-1, 30, 30)
    print(f"    Kin features: {', '.join(d['pf_features_kin_names'])}")
    sig = d['label'] == 1
    print(f"    QFIM as 30x30 matrices: max |F - F^T| = {np.abs(F - F.transpose(0, 2, 1)).max():.1e}; "
          f"median soft-drop mass signal / QCD = {np.median(d['jet_sdmass'][sig]):.0f} / {np.median(d['jet_sdmass'][~sig]):.0f} GeV")

## Models and weights

`part.build_model('ParT' | 'QUIVER', 'kin' | 'full')` builds the network with the constructor arguments of the thesis
runs (`get_model()` in `networks/example_ParticleTransformer.py` and, for QUIVER,
`networks/QFI-augmented/example_ParticleTransformer_qfim_simple.py` with `qfim_concat_mode='post_embed_particle_concat'`):
8 particle-attention + 2 class-attention blocks of width 128, 8 heads, pair embedding (64, 64, 64).
QUIVER embeds the 90 QFIM entries of every particle with its own embedding MLP (90 &rarr; 128 &rarr; 512 &rarr; 128) and
appends these 10 tokens to the 10 particle tokens; the ParT pair-interaction bias covers the particle block only.
Every weight file is the thesis `net_best_epoch_state.pt` of the run (as CPU tensors), loaded with `strict=True`.

In [ ]:
models = {}
for task in TASKS:
    for features in FEATURES:
        for budget in BUDGETS:
            for model in MODELS:
                name = f'{task.lower()}_{features.lower()}_{model.lower()}_{budget.lower()}'
                models[task, features, budget, model] = part.load_model(model, features, WEIGHTS / f'{name}.pt', DEVICE)

for features in FEATURES:
    for model in MODELS:
        net = models['TTbar', features, '100K', model]
        print(f'{model:6s} {features:4s}: {sum(p.numel() for p in net.parameters()):,} parameters')
print(f'loaded {len(models)} checkpoints from {WEIGHTS.relative_to(HERE)}/')

## Inference and AUC table

Scores are softmax(logits)[:, 1], computed in batches of 512 jets as in the thesis evaluation. For each checkpoint the
table lists the AUC on the demo jets with its bootstrap uncertainty, the thesis AUC of the same checkpoint on the full
test set (3.9 M jets), their difference, and the thesis mean &plusmn; std over the five seeds (Table of Chapter 6). The
last column compares the QUIVER &minus; ParT gain on the demo jets (same jets for both models, so it is much less noisy
than the individual AUCs) with the thesis gain of the same checkpoints.

In [ ]:
# thesis AUCs (figures/thesis_results/*.csv of the thesis repository):
# (task, features, N, model): (AUC of this checkpoint on the full test set, mean over seeds, std over seeds, seeds)
THESIS = {
    ('TTbar', 'Kin', '100K', 'ParT'): (0.971110, 0.971395, 0.000380, 5),
    ('TTbar', 'Kin', '100K', 'QUIVER'): (0.973531, 0.973683, 0.000135, 5),
    ('TTbar', 'Kin', '500K', 'ParT'): (0.976098, 0.976288, 0.000151, 5),
    ('TTbar', 'Kin', '500K', 'QUIVER'): (0.978849, 0.978482, 0.000447, 5),
    ('TTbar', 'Kin', '5M', 'ParT'): (0.978349, 0.978316, 0.000039, 5),
    ('TTbar', 'Kin', '5M', 'QUIVER'): (0.980695, 0.980699, 0.000033, 5),
    ('TTbar', 'Full', '100K', 'ParT'): (0.988275, 0.988391, 0.000237, 5),
    ('TTbar', 'Full', '100K', 'QUIVER'): (0.988431, 0.988527, 0.000223, 5),
    ('TTbar', 'Full', '500K', 'ParT'): (0.990719, 0.990721, 0.000077, 5),
    ('TTbar', 'Full', '500K', 'QUIVER'): (0.990939, 0.991006, 0.000178, 5),
    ('TTbar', 'Full', '5M', 'ParT'): (0.992351, 0.992355, 0.000028, 5),
    ('TTbar', 'Full', '5M', 'QUIVER'): (0.992474, 0.992435, 0.000026, 5),
    ('HToCC', 'Kin', '100K', 'ParT'): (0.944532, 0.943886, 0.000615, 5),
    ('HToCC', 'Kin', '100K', 'QUIVER'): (0.950147, 0.950474, 0.000252, 5),
    ('HToCC', 'Kin', '500K', 'ParT'): (0.953158, 0.953127, 0.000121, 5),
    ('HToCC', 'Kin', '500K', 'QUIVER'): (0.957924, 0.957669, 0.000352, 5),
    ('HToCC', 'Kin', '5M', 'ParT'): (0.955531, 0.955554, 0.000021, 5),
    ('HToCC', 'Kin', '5M', 'QUIVER'): (0.960560, 0.960538, 0.000023, 5),
    ('HToCC', 'Full', '100K', 'ParT'): (0.978322, 0.978446, 0.000137, 5),
    ('HToCC', 'Full', '100K', 'QUIVER'): (0.978527, 0.978599, 0.000128, 5),
    ('HToCC', 'Full', '500K', 'ParT'): (0.981357, 0.981246, 0.000111, 5),
    ('HToCC', 'Full', '500K', 'QUIVER'): (0.981320, 0.981396, 0.000067, 5),
    ('HToCC', 'Full', '5M', 'ParT'): (0.982979, 0.982965, 0.000070, 5),
    ('HToCC', 'Full', '5M', 'QUIVER'): (0.983387, 0.983494, 0.000325, 5),
}

def bootstrap_sigma(y, s, n=100, seed=0):
    rng = np.random.default_rng(seed)
    idx = [rng.integers(0, len(y), len(y)) for _ in range(n)]
    return float(np.std([roc_auc_score(y[i], s[i]) for i in idx]))

t0 = time.time()
auc, sigma = {}, {}
for key, net in models.items():
    task, features = key[0], key[1].lower()
    scores = part.predict(net, demo[task], features, batch_size=512, device=DEVICE)
    auc[key] = roc_auc_score(demo[task]['label'], scores)
    sigma[key] = bootstrap_sigma(demo[task]['label'], scores)
print(f'inference of {len(models)} models on {sum(len(d["label"]) for d in demo.values()):,} jets: {time.time() - t0:.0f} s\n')

head = (f"{'feat':4s} {'N':>4s} {'model':6s}  {'demo AUC':>16s}  {'thesis (rep 1)':>14s}  {'demo-thesis':>11s}  "
        f"{'thesis mean+-std (5 seeds)':>26s}  {'QUIVER-ParT demo / thesis':>25s}")
for task in TASKS:
    print(f'{TASKS[task]}\n{head}')
    for features in FEATURES:
        for budget in BUDGETS:
            for model in MODELS:
                k = (task, features, budget, model)
                ref, mean, std, n = THESIS[k]
                gain = ''
                if model == 'QUIVER':
                    p = (task, features, budget, 'ParT')
                    gain = f'{auc[k] - auc[p]:+.4f} / {ref - THESIS[p][0]:+.4f}'
                print(f'{features:4s} {budget:>4s} {model:6s}  {auc[k]:.4f} +- {sigma[k]:.4f}  {ref:14.5f}  '
                      f'{auc[k] - ref:+11.4f}  {mean:17.5f} +- {std:.5f}  {gain:>25s}')
    print()
dev = np.array([auc[k] - THESIS[k][0] for k in models])
pulls = dev / np.array([sigma[k] for k in models])
print(f'demo - thesis AUC over all {len(models)} checkpoints: mean {dev.mean():+.4f}, max |.| {np.abs(dev).max():.4f} '
      f'(max |pull| {np.abs(pulls).max():.1f} sigma)')

**Reading the table.** In the thesis (mean over five seeds) QUIVER improves the AUC over ParT with the Kin features at
every training budget, by about +0.002 for Top and +0.005 to +0.007 for H &rarr; cc&#772;, while with the Full feature
set the gain shrinks to a few 10<sup>&minus;4</sup>. On the demo jets the Kin gains of the individual checkpoints come
out close to their thesis values; the Full gains are below the resolution of 20,000 jets.

Most demo AUCs lie somewhat below the full-test values (by up to about 0.002, roughly one bootstrap &sigma;). This shift
is a property of these 20,000 jets, common to all checkpoints: the thesis's own (GPU) test scores of the same jets give
the demo AUCs to within 2&times;10<sup>&minus;6</sup>, and `weaver --predict` on the same jets reproduces the notebook
scores exactly (`tools/export_part.py verify`).